# From BED files to GPU kernels and PCA

PySnpTools can read into CuPy, standardize on the array's device, reuse fitted statistics, and construct kernels in SNP blocks. This notebook combines those operations with independently checked calculations. It uses the existing PySnpTools toy dataset, downloaded and cached by `example_file` on first use.

**Requirements:** matching development versions of PySnpTools and bed-reader, the bed-reader extension built with the **`cubecl-cuda` Cargo feature**, CuPy, a working CUDA driver, and an NVIDIA GPU (CUDA device 0). Also install Jupyter and Matplotlib in the notebook environment. These experimental features are not necessarily in published packages. See the [GPU guide](../source/gpu.rst) for supported options and limitations.

PySnpTools imports appear where they are first used. The capability check verifies the build; an actual CuPy operation verifies runtime access. A missing GPU is an error, so this notebook cannot quietly become a CPU demonstration.

In [28]:
from importlib.metadata import version
from time import perf_counter
from unittest.mock import patch

import cupy as cp
import matplotlib.pyplot as plt
import numpy as np
from bed_reader import has_gpu_support

if not has_gpu_support():
    raise RuntimeError("Rebuild bed-reader with the cubecl-cuda Cargo feature")
device = cp.cuda.runtime.getDeviceProperties(cp.cuda.Device().id)
print("GPU:", device["name"].decode())
print({package: version(package) for package in ("pysnptools", "bed-reader")}, "CuPy:", cp.__version__)

GPU: NVIDIA GeForce RTX 3060
{'pysnptools': '1.0.1', 'bed-reader': '1.2.1'} CuPy: 14.2.0


## Open an existing sample dataset

Use the same toy dataset as other PySnpTools examples. `example_file` downloads and caches the BED/BIM/FAM files together. We select 96 samples and 256 SNPs to keep the walkthrough quick; change or remove the selection to try more data.

BED stores allele counts (0/1/2). A separate, small in-memory example below demonstrates fractional dosages, missing values, and constant columns.


In [29]:
from pysnptools.snpreader import Bed
from pysnptools.util import example_file

bed_path = example_file("pysnptools/examples/toydata.*", "*.bed")
bed = Bed(bed_path, count_A1=True)
sample_count, snp_count = bed.iid_count, bed.sid_count
print(f"{sample_count} samples × {snp_count} SNPs")

500 samples × 10000 SNPs


## Select a backend, dtype, and memory order


In [30]:
# We use float32 because it is much, much faster on consumer GPUs than float64.

# Read on the CPU for comparison; reads otherwise default to float64.
cpu = bed.read(dtype="float32", xp=np)
snp_values, iid = cpu.val, cpu.iid

# Upload packed BED bytes and decode on the GPU.
# .val is a CuPy array backed by Rust-owned memory; its views retain ownership.
# F order stores each SNP's sample values together; C order stores each sample's SNP values together.
gpu = bed.read(dtype="float32", order="F", xp=cp)
assert isinstance(gpu.val, cp.ndarray) and gpu.val.flags.f_contiguous
# Sample IDs, SNP IDs, and positions stay on the CPU.
assert gpu.val.dtype == np.float32 and isinstance(gpu.iid, np.ndarray)
# Copy to the CPU only to check equality, including missing values (NaNs).
# Decoded allele counts are exact, so no floating-point tolerance is needed.
np.testing.assert_array_equal(cp.asnumpy(gpu.val), snp_values)

# Module names work too; this read uses float64 and C order.
# float64 is supported; its arithmetic can be much slower on consumer GPUs.
selected = bed[:4, :6].read(dtype="float64", order="C", xp="cupy")
assert selected.val.flags.c_contiguous and selected.val.dtype == np.float64
np.testing.assert_array_equal(cp.asnumpy(selected.val), snp_values[:4, :6])
np.testing.assert_array_equal(selected.iid, iid[:4])
print(type(gpu.val), gpu.val.dtype, "F order:", gpu.val.flags.f_contiguous)
print(type(selected.val), selected.val.dtype, "C order:", selected.val.flags.c_contiguous)


<class 'cupy.ndarray'> float32 F order: True
<class 'cupy.ndarray'> float64 C order: True


## Set a shared backend default


In [ ]:
from pysnptools.standardizer import Unit
from pysnptools.util import sub_matrix

# Everything inside uses this default where xp=None is supported,
# including reads inside other functions. Explicit xp still takes precedence.
# The environment is process-wide: avoid conflicting defaults in concurrent workflows.
with patch.dict("os.environ", {"ARRAY_MODULE": "cupy"}):
    device_default = bed[:4, :6].read(dtype="float32", xp=None)
    assert isinstance(device_default.val, cp.ndarray)
    # Omitting xp is equivalent to xp=None.
    also_device = bed[:4, :6].read(dtype="float32")
    assert isinstance(also_device.val, cp.ndarray)

    # Override the CuPy default for this read.
    host = bed[:4, :6].read(dtype="float32", xp=np)
    assert isinstance(host.val, np.ndarray)
# Exiting the block restores the previous environment, even on failure.

with patch.dict("os.environ", {"ARRAY_MODULE": "numpy"}):
    host_default = bed[:4, :6].read(dtype="float32")
    assert isinstance(host_default.val, np.ndarray)
    # Override the NumPy default for this read.
    device_copy = gpu.read(dtype="float32", xp=cp)
    # Standardization follows the existing array, regardless of the environment.
    device_copy.standardize(Unit())
    assert isinstance(device_copy.val, cp.ndarray)

# Without ARRAY_MODULE, xp=None defaults to NumPy.
# sub_matrix instead follows the input device; preserve float32 explicitly.
subset = sub_matrix(gpu.val, [7, 2, 7], [5, 3, 0], dtype=gpu.val.dtype, order="C")
assert isinstance(subset, cp.ndarray) and subset.dtype == np.float32
np.testing.assert_array_equal(cp.asnumpy(subset), snp_values[[7, 2, 7]][:, [5, 3, 0]])


## Fractional dosages: Unit and Beta

Unit uses each SNP's observed mean and population standard deviation (`ddof=0`), then fills missing values with zero. Final variance can be below one after imputation. Constant training columns become zero and store infinite standard deviation; fitting a wholly missing SNP raises `ValueError`.

Beta centers values and multiplies by a beta density at the minor-allele frequency; it **does not divide by the standard deviation**. Our Beta(2, 3) reference uses the polynomial density `12 p (1-p)²`, independently of PySnpTools and SciPy. Beta needs valid allele-frequency means. Unit also accepts negative and out-of-range finite values.

This deliberately small reference loops over SNPs using centered float64 residuals, avoiding raw-moment subtraction `E[x²] − E[x]²`. Tolerances are 32 machine epsilons for these small, bounded inputs; results need not be bit-identical across floating algorithms.

In [ ]:
from pysnptools.snpreader import SnpData
from pysnptools.standardizer import Beta

dosages = np.array([
    [0.25, 1.0, 0.5],
    [0.75, 1.0, np.nan],
    [np.nan, 1.0, 1.5],
    [1.75, 1.0, 1.0],
    [1.25, 1.0, 0.75],
], dtype=np.float64)
dosage_iid = [["family", str(sample)] for sample in range(len(dosages))]
dosage_sid = ["dosage", "constant", "missing"]

def reference(values, beta=False):
    result = np.zeros(values.shape, dtype=np.float64)
    stats = np.empty((values.shape[1], 2), dtype=np.float64)
    for column in range(values.shape[1]):
        observed = ~np.isnan(values[:, column])
        finite = values[observed, column].astype(np.float64)
        mean = finite.mean()
        deviation = np.sqrt(np.mean((finite - mean) ** 2))
        stats[column] = [mean, deviation if deviation != 0 else np.inf]
        if deviation == 0:
            continue
        if beta:
            frequency = min(mean / 2, 1 - mean / 2)
            factor = 12 * frequency * (1 - frequency) ** 2
        else:
            factor = 1 / deviation
        result[observed, column] = (finite - mean) * factor
    return result, stats

for dtype in (np.float32, np.float64):
    tolerance = 32 * np.finfo(dtype).eps
    for order in ("C", "F"):
        for standardizer in (Unit(), Beta(2, 3)):
            values = dosages.astype(dtype)
            if isinstance(standardizer, Unit):
                values[:, 0] -= 4  # Unit supports finite values outside [0, 2].
            expected, expected_stats = reference(values, beta=isinstance(standardizer, Beta))
            data = SnpData(iid=dosage_iid, sid=dosage_sid,
                           val=cp.array(values, order=order), xp=cp)
            _, trained = data.standardize(standardizer, return_trained=True)
            assert isinstance(data.val, cp.ndarray) and data.val.dtype == dtype
            assert data.val.flags["C_CONTIGUOUS" if order == "C" else "F_CONTIGUOUS"]
            assert isinstance(trained.stats, np.ndarray)
            np.testing.assert_allclose(cp.asnumpy(data.val), expected, rtol=tolerance, atol=tolerance)
            np.testing.assert_allclose(trained.stats, expected_stats, rtol=tolerance, atol=tolerance)
print("Fractional Unit/Beta checks passed for both floating dtypes and orders")

## Fit once; apply without refitting

Fit on training samples, then apply those moments to a held-out cohort. Only the small per-SNP statistics are returned to the CPU. The matrix stays on the GPU. Beta's per-SNP density evaluation uses SciPy on the CPU; that is different from transferring the matrix to a CPU standardizer.

UnitTrained can select and reorder known SNP IDs. BetaTrained requires the exact training IDs and order. Neither refits missing test columns or columns that were constant during training.

Trained statistics can also cross devices and floating dtypes. The CPU float64 application below uses the stored float32 moments; a wider target cannot restore precision already lost during training.

In [ ]:
training_rows = np.arange(0, sample_count, 2)
test_rows = np.arange(1, sample_count, 2)
training = bed[training_rows, :].read(dtype="float32", xp=cp)
_, fitted_unit = training.standardize(Unit(), return_trained=True)
test = bed[test_rows, :].read(dtype="float32", xp=cp).standardize(fitted_unit)
assert isinstance(test.val, cp.ndarray) and isinstance(fitted_unit.stats, np.ndarray)

# An independent application of the stored moments, not a fit on test rows.
means, deviations = fitted_unit.stats.T
expected_test = (snp_values[test_rows].astype(np.float64) - means) / deviations
expected_test[np.isnan(snp_values[test_rows])] = 0
float32_tolerance = 32 * np.finfo(np.float32).eps
np.testing.assert_allclose(cp.asnumpy(test.val), expected_test,
                           rtol=float32_tolerance, atol=float32_tolerance)
reordered = bed[test_rows, [5, 3, 0]].read(dtype="float32", xp=cp).standardize(fitted_unit)
np.testing.assert_array_equal(cp.asnumpy(reordered.val), cp.asnumpy(test.val)[:, [5, 3, 0]])
print("Stored moments:", fitted_unit.stats.shape, fitted_unit.stats.dtype)

# Reuse GPU-trained float32 moments on a CPU float64/F-order target.
host_test = bed[test_rows, :].read(dtype="float64", order="F", xp=np).standardize(fitted_unit)
assert isinstance(host_test.val, np.ndarray) and host_test.val.flags.f_contiguous
np.testing.assert_allclose(host_test.val, expected_test,
                           rtol=8 * np.finfo(np.float64).eps,
                           atol=8 * np.finfo(np.float64).eps)

## Build kernels in SNP blocks, reusing Beta training statistics

A kernel is `Z @ Z.T`, where `Z` contains standardized SNP values. Native BED kernel construction reads and standardizes on the GPU. A positive integer `block_size` bounds each SNP block, **not the dense sample-by-sample kernel or all temporary allocations**. It is not an out-of-core eigensolver.

We train Beta on one cohort and build the held-out kernel with those exact moments. PySnpTools validates the complete SNP order before using the matching trained statistics for each block. We also compare with an independent float64 product, explicitly allowing float32 accumulation roundoff: 64 epsilons relative, and 64 epsilons times the reference absolute-product scale for the absolute error.

In [ ]:
beta_training = bed[training_rows, :].read(dtype="float32", xp=cp)
_, fitted_beta = beta_training.standardize(Beta(2, 3), return_trained=True)
means, deviations = fitted_beta.stats.T
frequency = np.minimum(means.astype(np.float64) / 2, 1 - means.astype(np.float64) / 2)
weights = 12 * frequency * (1 - frequency) ** 2
weights[np.isinf(deviations)] = 0
expected_beta = (snp_values[test_rows].astype(np.float64) - means) * weights
expected_beta[np.isnan(snp_values[test_rows])] = 0
expected_kernel = expected_beta @ expected_beta.T
absolute_product_scale = np.max(np.abs(expected_beta) @ np.abs(expected_beta).T)
kernel_tolerance = 64 * np.finfo(np.float32).eps

beta_kernel = bed[test_rows, :].read_kernel(
    fitted_beta, block_size=37, dtype="float32", order="C", xp=cp)
assert isinstance(beta_kernel.val, cp.ndarray) and beta_kernel.val.flags.c_contiguous
np.testing.assert_allclose(cp.asnumpy(beta_kernel.val), expected_kernel,
                           rtol=kernel_tolerance,
                           atol=kernel_tolerance * absolute_product_scale)
unit_kernel = bed.read_kernel(Unit(), block_size=37, dtype="float64", order="F", xp=cp)
expected_unit, _ = reference(snp_values)
np.testing.assert_allclose(cp.asnumpy(unit_kernel.val), expected_unit @ expected_unit.T,
                           rtol=64 * np.finfo(np.float64).eps,
                           atol=64 * np.finfo(np.float64).eps * snp_count)
print("Beta test kernel:", beta_kernel.val.shape, beta_kernel.val.dtype)
print("Unit full-cohort kernel:", unit_kernel.val.shape, unit_kernel.val.dtype)

## Normalize and perform PCA on the GPU

`DiagKtoN` scales the whole kernel to have average diagonal one; Unit standardization alone does not promise that after missing-value imputation. Use a finite, positive-trace kernel. Normalization and the eigendecomposition operate on CuPy arrays. Only the two plotting coordinates and a few eigenvalues are copied back.

The plot shows the toy samples without assigning population labels; it is a demonstration, not a biological finding. Eigenvector signs are arbitrary; the sign of an axis can change between valid runs.

In [ ]:
from pysnptools.standardizer import DiagKtoN

unit_kernel.standardize(DiagKtoN())
trace = float(cp.trace(unit_kernel.val))
np.testing.assert_allclose(trace, sample_count, rtol=64 * np.finfo(np.float64).eps, atol=0)
eigenvalues, eigenvectors = cp.linalg.eigh(unit_kernel.val)
# Check the two eigenpairs before copying their small plotting coordinates.
residual = unit_kernel.val @ eigenvectors[:, -2:] - eigenvectors[:, -2:] * eigenvalues[-2:]
relative_residual = float(cp.linalg.norm(residual) / cp.linalg.norm(unit_kernel.val))
assert relative_residual < 1e-12  # float64 eigensolver, small well-scaled matrix
coordinates = cp.asnumpy(eigenvectors[:, [-1, -2]] * cp.sqrt(eigenvalues[[-1, -2]]))
figure, axes = plt.subplots(figsize=(6, 4))
axes.scatter(coordinates[:, 0], coordinates[:, 1])
axes.set(xlabel="PC1", ylabel="PC2", title="GPU kernel PCA on the toy dataset")
figure.tight_layout()
plt.show()
print("Largest eigenvalues:", cp.asnumpy(eigenvalues[-3:][::-1]))

## Time completed work

CuPy launches are asynchronous. Synchronize before and after a wall-clock measurement so the interval includes completed work. Warm up the exact operation first; this excludes first-use compilation from the reported time. This measures a complete local-file read/standardize/kernel pipeline, including host work and synchronization. It is not isolated kernel time or a claim that this small example is faster than the CPU.

Use representative data and repeated measurements for a performance study. Native decoding and CuPy may use different streams; device-wide synchronization covers both.

In [ ]:
def timed_kernel_read():
    return bed.read_kernel(Unit(), block_size=37, dtype="float32", xp=cp)

warmup = timed_kernel_read()
cp.cuda.runtime.deviceSynchronize()
del warmup
elapsed = []
for repetition in range(3):
    cp.cuda.runtime.deviceSynchronize()
    started = perf_counter()
    measured_kernel = timed_kernel_read()
    cp.cuda.runtime.deviceSynchronize()
    elapsed.append(perf_counter() - started)
    assert isinstance(measured_kernel.val, cp.ndarray)
    del measured_kernel
print(f"Completed GPU pipeline, median of 3 warm runs: {np.median(elapsed) * 1000:.2f} ms")

The same explicit-backend and trained-statistics choices carry into FaST-LMM association workflows; see its GPU guide for worker execution and backend selection. This notebook stops at PySnpTools' public kernel API.

CPU reads remain available with `xp=np`. Integer BED reads are supported with the documented opt-in, but int8 standardization is rejected; read or convert to float32/float64 first. Other file formats may read on the CPU and then transfer. GPU availability errors are not silently retried on the CPU.

The downloaded sample files stay cached for reuse. Device arrays own their decoded data. Dropping arrays releases ownership, but allocation pools can retain memory for reuse; deleting a name is not a guarantee that another GPU library immediately receives that memory.